# 대구 교통사고 × 인구 EDA

사고 데이터와 구군 연령별 인구를 탐색하고, 위험도 모델 피처·타깃 분포를 확인합니다.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
sns.set_theme(style="whitegrid", font="Malgun Gothic")

BASE = Path("..").resolve()
ACC_PATH = BASE / "data" / "raw" / "사고분석.csv"
POP_PATH = BASE / "data" / "raw" / "대구_구군_연령별_주민등록인구_2020_2025.csv"
RATE_PATH = BASE / "data" / "processed" / "인구대비_가중사고비율.csv"

acc = pd.read_csv(ACC_PATH, encoding="utf-8")
acc["구군"] = acc["시군구"].astype(str).str.replace(r"^대구광역시\s*", "", regex=True)
print("사고 shape:", acc.shape)
acc.head()

## 1. 기본 정보 · 결측

In [ ]:
display(acc.info())
nulls = acc.isnull().sum()
nulls = nulls[nulls > 0].sort_values(ascending=False)
print("결측 열:")
display(nulls.to_frame("null_count"))

## 2. 타깃(`사고내용`) · 클래스 불균형

In [ ]:
target_cnt = acc["사고내용"].value_counts()
target_pct = acc["사고내용"].value_counts(normalize=True).mul(100).round(2)
display(pd.DataFrame({"건수": target_cnt, "비율(%)": target_pct}))

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=target_cnt.index, y=target_cnt.values, ax=ax, color="#3d5a80")
ax.set_title("사고내용(경중) 분포")
ax.set_ylabel("건수")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 3. 모델 입력 피처 분포

위험도 모델 피처: `구군`, `가해운전자 연령대`, `가해운전자 성별`, `가해운전자 차종`, `주야`

In [ ]:
feature_cols = ["구군", "가해운전자 연령대", "가해운전자 성별", "가해운전자 차종", "주야"]

fig, axes = plt.subplots(3, 2, figsize=(12, 12))
axes = axes.ravel()
for i, col in enumerate(feature_cols):
    vc = acc[col].value_counts()
    sns.barplot(y=vc.index.astype(str), x=vc.values, ax=axes[i], color="#98c1d9")
    axes[i].set_title(col)
    axes[i].set_xlabel("건수")
axes[-1].axis("off")
plt.tight_layout()
plt.show()

## 4. 피처 × 사고내용 교차 (위험도 관점)

In [ ]:
severity = ["사망사고", "중상사고", "경상사고", "부상신고사고"]

def severity_rate(df, by):
    ct = pd.crosstab(df[by], df["사고내용"], normalize="index")
    for c in severity:
        if c not in ct.columns:
            ct[c] = 0.0
    return ct[severity]

for col in ["구군", "가해운전자 연령대", "가해운전자 차종", "주야"]:
    rate = severity_rate(acc, col)
    ax = rate.plot(kind="barh", stacked=True, figsize=(10, max(3, len(rate) * 0.35)),
                   colormap="RdYlGn_r")
    ax.set_title(f"{col}별 사고내용 비율")
    ax.set_xlabel("비율")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

## 5. 시계열 · 구군 추이

In [ ]:
def to_period(s):
    t = str(s).replace(" ", "")
    year = t.split("년")[0]
    month = int(t.split("년")[1].replace("월", ""))
    return f"{year}.{'1/2' if month <= 6 else '2/2'}"

acc["기간"] = acc["발생년월"].map(to_period)
monthly = acc.groupby("발생년월").size()
fig, ax = plt.subplots(figsize=(12, 3.5))
monthly.plot(ax=ax, color="#293241")
ax.set_title("월별 사고 건수")
ax.set_ylabel("건수")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

by_gu = acc.groupby(["기간", "구군"]).size().unstack(fill_value=0)
by_gu.plot(figsize=(12, 4), title="반기·구군별 사고 건수")
plt.ylabel("건수")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## 6. 인구 데이터 · 인구 대비 가중 사고비율

In [ ]:
pop_raw = pd.read_csv(POP_PATH, header=None, encoding="utf-8")
print("인구 CSV shape:", pop_raw.shape)
print("행정구역:", pop_raw.iloc[2:, 0].unique().tolist())

if RATE_PATH.exists():
    rate = pd.read_csv(RATE_PATH, encoding="utf-8-sig")
    display(rate.head())
    top = (
        rate.groupby("구군", as_index=False)["인구10만당_가중"]
        .mean()
        .sort_values("인구10만당_가중", ascending=False)
    )
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(data=top, x="인구10만당_가중", y="구군", ax=ax, color="#ee6c4d")
    ax.set_title("구군별 인구 10만당 가중사고점수 (기간 평균)")
    plt.tight_layout()
    plt.show()
else:
    print("인구대비_가중사고비율.csv 없음 → python -m src.preprocess 실행 후 확인")

## 7. 모델용 정제 데이터 요약

`기타불명`·결측 제거 후 학습에 들어가는 규모를 확인합니다.

In [ ]:
cols = feature_cols + ["사고내용"]
use = acc[cols].dropna()
for c in feature_cols:
    use = use[~use[c].astype(str).str.contains("기타불명", na=False)]
print(f"원본 {len(acc):,} → 모델용 {len(use):,} ({len(use)/len(acc)*100:.1f}%)")
display(use["사고내용"].value_counts(normalize=True).mul(100).round(2).to_frame("비율(%)"))
use.describe(include="all").T

## 8. EDA 요약

- **타깃 불균형**: 경상사고 비중이 압도적으로 큼 → accuracy만 보면 ‘전부 경상’ 모델이 유리.
- **지역**: 달서구·수성구·북구 사고 건수 상위. 인구 대비 지표는 `data/processed/인구대비_가중사고비율.csv` 참고.
- **차종**: 승용이 다수. 이륜·PM 등은 건수는 적지만 경중 비율이 다를 수 있음.
- **모델 피처**: 상세 정의는 `README.md` 참고.